# e1155885 — LightGBM masked ensemble

This notebook documents data processing, feature engineering, the model recipe, temporal prediction validation, and generation of `submissions/e1155885.csv` for BT4012 Competition 2026.

The retained ensemble (formerly v16) achieved a **user-reported public ROC-AUC of 0.95258** on 21 October 2026. It blends **50% fixed anchor predictions** with **50% the mean of three 6,000-tree LightGBM models**, using seeds 2026, 17, and 73. The anchor originated from the earlier v13 ensemble and is included as a fixed prediction asset.

Run all cells in order. Default execution reloads the checked-in final models, validates historical prediction snapshots, and generates the CSV. Optional core retraining is available below; rebuilding the historical anchor ensemble is outside this retained release.

## 1. Setup and repository paths

Install `requirements.txt`, launch this notebook from the repository root, and select the matching Python kernel. Put the four competition CSV files in `bt-4012-competition-2026/`. The data is excluded from Git; model assets and historical prediction snapshots are included. No Kaggle credentials are needed by the notebook.

In [1]:
from pathlib import Path
import json
import sys
import time
import importlib.metadata
import os
import tempfile

# Use a writable cache when optional plotting libraries are imported by LightGBM.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "bt4012-matplotlib"))

ROOT = Path.cwd().resolve()
if not (ROOT / "fraud_pipeline").is_dir():
    raise RuntimeError("Open and run this notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from fraud_pipeline.data import load_data
from fraud_pipeline.notebook_workflow import (
    ASSETS, check_data, fit_core, load_release, make_submission,
    masked_training_arrays, sha256, validation_table,
)

DATA_DIR = ROOT / "bt-4012-competition-2026"
OUTPUT_CSV = ROOT / "submissions/e1155885.csv"
REFIT_CORE = False  # True fits all three 6,000-tree core models; can take a long time.
release = load_release()
print("Release:", release["name"])
print("Checked release assets:", len(release["files"]))
print({p: importlib.metadata.version(p) for p in
       ["numpy", "pandas", "lightgbm", "scikit-learn", "joblib"]})

Release: lgbm_masked_ensemble
Checked release assets: 37
{'numpy': '2.0.2', 'pandas': '2.3.3', 'lightgbm': '4.6.0', 'scikit-learn': '1.6.1', 'joblib': '1.5.3'}


## 2. Data processing and input validation

Read training labels `none` and `unknown` as missing values. Only observed labels 0 and 1 enter supervised fitting. Validate transaction IDs, feature schema, time ordering, sample submission indices, and original input hashes before making predictions. The graph edge list is loaded for input auditing; the selected release does not use graph-derived features.

In [2]:
check_data(DATA_DIR, release)
train, test, edges, sample, features = load_data(DATA_DIR)
labeled = train.loc[train.label.notna()].reset_index(drop=True)
assert labeled.label.isin([0, 1]).all()
assert labeled.time_step.max() < test.time_step.min()
assert set(labeled.txId).isdisjoint(test.txId)
print(pd.DataFrame([
    {"split": "train (all)", "rows": len(train), "time_min": train.time_step.min(), "time_max": train.time_step.max()},
    {"split": "train (labeled)", "rows": len(labeled), "time_min": labeled.time_step.min(), "time_max": labeled.time_step.max()},
    {"split": "test", "rows": len(test), "time_min": test.time_step.min(), "time_max": test.time_step.max()},
]).to_string(index=False))
print("Missing-label rows excluded from supervised fitting:", int(train.label.isna().sum()))
print("Labeled class counts:", labeled.label.value_counts().sort_index().to_dict())
print("Supplied graph edges:", len(edges))
print("Anonymous feature columns:", len(features))

          split   rows  time_min  time_max
    train (all) 141772         1        35
train (labeled)  31235         1        35
           test  15329        36        49
Missing-label rows excluded from supervised fitting: 110537
Labeled class counts: {0.0: 27591, 1.0: 3644}
Supplied graph edges: 234355
Anonymous feature columns: 165


## 3. Feature engineering and leakage controls

Use the **165 supplied anonymous numeric features plus `time_step`**. Drop transaction IDs, submission indices, and the target from the feature matrix. Anonymous feature meanings are not inferred, and identifiers do not enter the model. For each fit cutoff, calculate medians only on labeled fitting rows, replace missing values using those medians, and use `float32` matrices. Tree models do not require standardization.

During training, create two additional views by independently replacing 75% of anonymous feature entries with missing values, keeping `time_step` intact. Missing values in those views are handled natively by LightGBM. Allocate 25% of each row's weight to the original view and 37.5% to each masked view, with an additional positive-class weight of 8. The masking is training augmentation; inference uses the original feature matrix.

In [3]:
feature_columns = features + ["time_step"]
assert len(feature_columns) == 166
assert not {"txId", "index", "label", "target"}.intersection(feature_columns)
train_features = labeled[feature_columns].astype("float32")
training_medians = train_features.median().fillna(0)
x_train = train_features.fillna(training_medians)
x_test = test[feature_columns].astype("float32").fillna(training_medians)
assert x_train.notna().all().all() and x_test.notna().all().all()
print("Training matrix:", x_train.shape, "Test matrix:", x_test.shape)
print("Dtype:", str(x_train.dtypes.iloc[0]))

# Small demonstration of the exact training augmentation; this does not fit a model.
demo_x, demo_y, demo_weights = masked_training_arrays(
    x_train.iloc[:8], labeled.label.iloc[:8], release["config"], release["seeds"][0]
)
assert len(demo_x) == 3 * 8
assert demo_x.time_step.notna().all()
np.testing.assert_allclose(
    demo_weights.reshape(3, 8).sum(axis=0),
    np.where(labeled.label.iloc[:8].to_numpy() == 1, 8.0, 1.0),
)
print("Masked training demo shape:", demo_x.shape)
print("Missing anonymous-feature cells in demo:", int(demo_x[features].isna().sum().sum()))

Training matrix: (31235, 166) Test matrix: (15329, 166)
Dtype: float32
Masked training demo shape: (24, 166)
Missing anonymous-feature cells in demo: 1963


## 4. Model specification and optional core training

The frozen core configuration uses 6,000 trees, learning rate 0.01, 31 leaves, L1 penalty 0.2, L2 penalty 5, feature fraction 0.85, bagging fraction 0.85, and deterministic CPU training. The three seeds are averaged equally. Default execution loads the retained fitted models and checks the saved feature schema, medians, seed, tree count, and configuration.

Set `REFIT_CORE = True` in the setup cell to train this core recipe on labeled periods through 35. This can take substantial time. Refitting retains the fixed anchor; it does not rebuild that earlier ensemble. The retained historical validation snapshots always describe the released models, even when the optional core refit is enabled. Numerical results from retraining may vary by platform.

In [4]:
print("Core configuration:", json.dumps(release["config"], indent=2))
print("Seeds:", release["seeds"])
print("Core blend weight:", release["weight"])
models = []
with threadpool_limits(limits=2):
    for seed in release["seeds"]:
        if REFIT_CORE:
            started = time.monotonic()
            bundle = fit_core(release["config"], labeled, features, seed, threads=2)
            print(f"Refitted seed {seed} in {time.monotonic() - started:.1f}s")
        else:
            path = ASSETS / f"models/long_emphasized8__seed{seed}__35.joblib"
            bundle = joblib.load(path)
            print("Loaded", path.name)
        assert bundle["columns"] == feature_columns
        assert bundle["seed"] == seed and bundle["config"] == release["config"]
        assert bundle["model"].current_iteration() == release["trees"]
        pd.testing.assert_series_equal(bundle["medians"], training_medians)
        models.append(bundle)

Core configuration: {
  "rate": 0.75,
  "copies": 2,
  "original_mass": 0.25,
  "leaves": 31,
  "positive_weight": 8.0,
  "trees": 6000,
  "checkpoints": [
    3000,
    6000
  ],
  "retain_time": true,
  "extra_trees": false,
  "l2": 5.0,
  "feature_fraction": 0.85,
  "name": "long_emphasized8"
}
Seeds: [2026, 17, 73]
Core blend weight: 0.5
Loaded long_emphasized8__seed2026__35.joblib
Loaded long_emphasized8__seed17__35.joblib
Loaded long_emphasized8__seed73__35.joblib


## 5. Temporal prediction validation

Recompute ROC-AUC from the checked-in out-of-time prediction snapshots. Every scored transaction must occur after the corresponding fitting cutoff, and its label and time must match the supplied training data. Blend the saved core predictions with the historical anchor predictions and compare each AUC to the frozen release receipt.

The first six windows were used for development; two additional windows evaluated the frozen choice. All these periods were previously inspected and many windows overlap. They are **reused development results, not an untouched holdout**, and their mean does not establish hidden-test performance. The public score is reported separately; no test labels are available. This step checks prediction snapshots rather than refitting historical estimators.

In [5]:
validation = validation_table(train, release)
print(validation.to_string(index=False, float_format=lambda value: f"{value:.8f}"))
development = validation.iloc[:6]
additional = validation.iloc[6:]
np.testing.assert_allclose(development.ensemble_auc.mean(), release["development_mean_auc"], atol=1e-12, rtol=0)
print(f"Development mean ROC-AUC: {development.ensemble_auc.mean():.10f}")
print(f"Development mean gain over anchor: {development.gain.mean():+.10f}")
print(f"Additional-window mean gain: {additional.gain.mean():+.10f}")
print("User-reported public ROC-AUC:", release["public_result"]["public_score"])

 fit_through  score_from  score_to  anchor_auc  ensemble_auc        gain
          14          15        28  0.96566317    0.96544628 -0.00021690
          21          22        35  0.91270947    0.91424882  0.00153935
          23          24        35  0.98833293    0.98791630 -0.00041663
          28          29        35  0.99751237    0.99762363  0.00011127
          17          18        31  0.93047890    0.93190069  0.00142179
          20          21        34  0.90722708    0.90949579  0.00226870
          18          19        32  0.91282988    0.91479596  0.00196608
          19          20        33  0.91756510    0.92045080  0.00288569
Development mean ROC-AUC: 0.9511052500
Development mean gain over anchor: +0.0007845956
Additional-window mean gain: +0.0024258860
User-reported public ROC-AUC: 0.95258


## 6. Test prediction and ensemble construction

Apply each fitted core estimator to the original test features using its own fitting-only medians. Average the three core probability vectors, align the fixed anchor by submission `index`, and form `0.5 * anchor + 0.5 * core_mean`. The fixed anchor restricts this release to the supplied competition test set.

In [6]:
core_predictions = []
with threadpool_limits(limits=2):
    for bundle in models:
        matrix = test[bundle["columns"]].astype("float32").fillna(bundle["medians"])
        probability = bundle["model"].predict(matrix, num_iteration=release["trees"], num_threads=2)
        assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
        core_predictions.append(probability)
core_mean = np.mean(core_predictions, axis=0)
anchor = pd.read_csv(ASSETS / "dependencies/v13_anchor.csv")
assert anchor["index"].is_unique and set(anchor["index"]) == set(test["index"])
anchor_probability = anchor.set_index("index").target.loc[test["index"]].to_numpy()
final_probability = (1 - release["weight"]) * anchor_probability + release["weight"] * core_mean
print("Prediction count:", len(final_probability))
print("Probability range:", float(final_probability.min()), float(final_probability.max()))

Prediction count: 15329
Probability range: 7.797243951124717e-06 0.9999444751980713


## 7. Generate and validate the submission CSV

Write exactly `index,target`, matching every sample-submission ID once and in the same order. Check finite probabilities in [0, 1], reload the written CSV, and compare it to the retained original submission when using the saved models. Only probabilities are submitted; no decision threshold is applied. Small floating-point differences may change the CSV's byte hash while preserving equivalent probabilities.

In [7]:
submission = make_submission(sample, test, final_probability, OUTPUT_CSV)
assert len(submission) == len(sample) == 15329
assert submission["index"].is_unique
assert np.isfinite(submission.target).all() and submission.target.between(0, 1).all()
if not REFIT_CORE:
    original = pd.read_csv(ASSETS / "submission.csv")
    np.testing.assert_array_equal(submission["index"], original["index"])
    np.testing.assert_allclose(submission.target, original.target, atol=1e-12, rtol=0)
    print("Maximum error vs original release:", float(np.abs(submission.target - original.target).max()))
print(submission.head().to_string(index=False))
print("Wrote:", OUTPUT_CSV.relative_to(ROOT))
print("SHA-256:", sha256(OUTPUT_CSV))

Maximum error vs original release: 4.440892098500626e-16
 index   target
     0 0.000592
     1 0.008262
     2 0.000963
     3 0.000457
     4 0.000076
Wrote: submissions/e1155885.csv
SHA-256: fc109588f30018cd82d209bb11e7d7202c975080c8c2b2f839c846b312517c2f


The notebook generates the submission locally and does not upload it to Kaggle. Model assets are included in the repository; the competition data, local virtual environment, older full study artifacts, and generated submissions are excluded by `.gitignore`.